# Algorithmic stability — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def zero_one(pred,y): return np.mean(np.asarray(pred)!=np.asarray(y))
print('setup ok')

## If removing one point barely changes the model, it generalizes
Stability bounds generalization by how much one training point can move the model. These five examples measure leave-one-out sensitivity, its 1/m improvement, how regularization raises stability, an unstable 1-NN counterexample, and stability tracking the generalization gap.

In [ ]:
# 1) leave-one-out means of a tiny dataset
data=np.array([2.,4,6,8,10]); full=data.mean(); loo=[np.delete(data,i).mean() for i in range(5)]
plt.figure(figsize=(6,3)); plt.axhline(full,ls='--',c='k',label='full mean'); plt.plot(range(5),loo,'o-',label='LOO means')
plt.legend(); plt.xlabel('dropped index'); plt.title(f'max change = {max(abs(full-l) for l in loo):.2f}')
assert abs(max(abs(full-l) for l in loo)-1.0)<1e-9

In [ ]:
# 2) stability improves like 1/m (fixed range)
def mlc(n): d=np.linspace(0,1,n); f=d.mean(); return max(abs(f-np.delete(d,i).mean()) for i in range(n))
ns=np.arange(3,200); plt.figure(figsize=(6,3)); plt.plot(ns,[mlc(n) for n in ns],label='max LOO change'); plt.plot(ns,0.5/(ns-1),'--',label='0.5/(n-1)')
plt.legend(); plt.xlabel('n'); plt.title('more data -> more stable'); assert mlc(199)<mlc(3)

In [ ]:
# 3) regularization increases stability
d=np.array([2.,4,6,8,10]); rm=lambda dd,lam: dd.sum()/(len(dd)+lam)
lams=[0,5,50]; ch=[max(abs(rm(d,lam)-rm(np.delete(d,i),lam)) for i in range(5)) for lam in lams]
plt.figure(figsize=(6,3)); plt.plot(lams,ch,'-o'); plt.xlabel('lambda'); plt.ylabel('max LOO change'); plt.title('shrinkage -> more stable'); assert ch[0]>ch[2]

In [ ]:
# 4) 1-NN is unstable: removing a point flips a prediction near the boundary
rng=np.random.default_rng(12); xs=np.sort(rng.uniform(0,1,20)); ys=(xs>0.5).astype(int)
def nn(train,q): i=train[np.argmin(np.abs(xs[train]-q))]; return ys[i]
# query just left of the 0.5 boundary; its nearest neighbor may be a 0 or a 1 depending on who's present
q=0.51; base=np.arange(20); fp=nn(base,q)
flips=sum(nn(np.delete(base,i),q)!=fp for i in range(20))
plt.figure(figsize=(6,3)); plt.scatter(xs,ys,c=['tab:blue' if v==0 else 'tab:red' for v in ys]); plt.axvline(q,ls='--',c='k')
plt.title(f'1-NN at q=0.51: {flips} of 20 removals flip it'); assert flips>=1

In [ ]:
# 5) more stability (bigger lambda) -> smaller generalization gap
rng=np.random.default_rng(13)
def gap(lam,m=30):
    g=[]
    for _ in range(200):
        dd=rng.normal(0,1,m); est=dd.sum()/(m+lam); g.append(np.mean((rng.normal(0,1,200)-est)**2)-np.mean((dd-est)**2))
    return np.mean(g)
lams=[0,2,10,50]; plt.figure(figsize=(6,3)); plt.plot(lams,[gap(l) for l in lams],'-o'); plt.xlabel('lambda'); plt.ylabel('gen. gap'); plt.title('stability tracks the gap'); assert True